# ISS Group 24 Modeling

**Before running:**
1. Open the shared Drive folder link and click **"Add shortcut to Drive"** → place it in *My Drive* root, keeping the name `iss_group_24`.
2. Select a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it automatically via checkpoint detection.

In [ ]:
# ── Standard library imports ─────────────────────────────────────────────────
import os
import sys
import subprocess
from pathlib import Path

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/iss_group_24"
REPO_URL        = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"

MANIFEST  = f"{DRIVE_ROOT}/dataset/cleaned/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/cleaned"
MODEL_DIR = f"{DRIVE_ROOT}/model"

In [ ]:
# ── Setup functions ──────────────────────────────────────────────────────────
from google.colab import drive


def mount_drive() -> None:
    """Mount Google Drive at /content/drive."""
    drive.mount("/content/drive")
    assert Path(DRIVE_ROOT).exists(), (
        f"Drive folder not found at {DRIVE_ROOT}.\n"
        "Make sure you added a shortcut to 'iss_group_24' in the root of My Drive."
    )
    print(f"Drive mounted. Project root: {DRIVE_ROOT}")


def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab (torch/torchvision are pre-installed)."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy", "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")

In [ ]:
# ── Run setup ────────────────────────────────────────────────────────────────
mount_drive()
setup_repo()
install_deps()

# Set CWD to the Drive project root so that relative paths written by
# train.py (analysis/, manifest parent, etc.) land on the Drive.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Project imports (available after setup_repo adds REPO_LOCAL_PATH) ────────
import aggregator
from modeling.train import train

---
## Step 0 — Build Dataset Manifest

Copies and indexes all images from `dataset/original/` into `dataset/cleaned/`.  
**Run once.** Skip if `manifest.json` already exists.

In [ ]:
if Path(MANIFEST).exists():
    print(f"Manifest already exists at {MANIFEST} — skipping aggregator.")
else:
    print("Running aggregator (this copies images to dataset/cleaned/ — may take several minutes)…")
    aggregator.main()

---
## Training

Shared kwargs used by every stage cell. Each stage cell overrides only the relevant epoch count; the resume mechanism automatically skips already-completed stages.

In [ ]:
# ── Shared training configuration ────────────────────────────────────────────
TRAIN_KWARGS = dict(
    manifest         = MANIFEST,
    data_root        = DATA_ROOT,
    out_dir          = MODEL_DIR,
    episodes_per_epoch = 1000,
    val_episodes     = 200,
    batch_size       = 16,
    num_workers      = 2,   # Drive FUSE I/O is the bottleneck; 2 workers is sufficient
    resume           = True,  # auto-resume from model/last.pt; safe even without a checkpoint
    # all stage epochs off by default — each cell turns on exactly one
    phase1_frozen_epochs  = 0,
    phase1_partial_epochs = 0,
    phase2_frozen_epochs  = 0,
    phase2_partial_epochs = 0,
    phase2_full_epochs    = 0,
)

### Stage 1.1 — VizWiz base · backbone frozen · heads LR 1e-3

Warms up the matcher heads (SupportTokenizer + CrossAttentionHead + DetHead) on 100 diverse VizWiz categories using the rotation-synthesis episode scheme. Backbone stays frozen — no ImageNet features are modified yet.

In [ ]:
train(**{**TRAIN_KWARGS, "phase1_frozen_epochs": 10})

### Stage 1.2 — VizWiz base · features[7:] @ 1e-5 · heads LR 5e-4

Partially unfreezes the upper MobileNetV3 blocks so the backbone can adapt its feature representations to the VizWiz scene domain while pretraining on diverse categories.

In [ ]:
train(**{**TRAIN_KWARGS, "phase1_partial_epochs": 20})

### Stage 2.1 — HOTS + InsDet + VizWiz novel · backbone re-frozen · heads LR 5e-4

Domain transfer warmup. Backbone is re-frozen to stabilize features while the matcher heads adapt from VizWiz pretraining to the target product/scene domain. Val switches to the HOTS+InsDet split.

In [ ]:
train(**{**TRAIN_KWARGS, "phase2_frozen_epochs": 10})

### Stage 2.2 — target domain · features[7:] @ 1e-5 · heads LR 5e-4

Main fine-tuning stage. Hard-negative mining is on (ratio 0.5). Contrastive weight bumped to 0.2. attn_loss_weight=1.0 (HOTS/InsDet have meaningful bbox attention targets).

In [ ]:
train(**{**TRAIN_KWARGS, "phase2_partial_epochs": 35})

### Stage 2.3 — full unfreeze · all backbone @ 5e-6 · heads LR 1e-4

Fine-tunes the entire network end-to-end at very low LR. Lower backbone layers stay at 5e-6 to prevent catastrophic forgetting of low-level ImageNet features.

In [ ]:
train(**{**TRAIN_KWARGS, "phase2_full_epochs": 10})